# Project – Emotion Classifier (GoEmotions)
Goal: Fine-tune DistilBERT for emotion detection and save results for later combined analysis.

## 1. Setup
- Import libraries
- Check GPU
- Fix random seed

In [1]:
import os
import json
import random
import numpy as np

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

# Paths
PROJECT_ROOT = r"E:\JYU\NLP\NLP\Project"
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.5.1+cu121
cuda available: True
gpu: NVIDIA GeForce GTX 1650


## 2. Data
- Load GoEmotions dataset (HuggingFace)
- Keep a manageable subset for quick runs (for now)
- Prepare label mapping

In [ ]:
ds = load_dataset("go_emotions")

print(ds)
print("train size:", len(ds["train"]))
print("validation size:", len(ds["validation"]))
print("test size:", len(ds["test"]))

# Small dev subset
train_small = ds["train"].shuffle(seed=SEED).select(range(20000))
val_small   = ds["validation"].shuffle(seed=SEED).select(range(2000))
test_small  = ds["test"].shuffle(seed=SEED).select(range(2000))

label_names = ds["train"].features["labels"].feature.names
num_labels = len(label_names)

print("num_labels:", num_labels)
print("example labels:", label_names[:10])
print("sample row:", train_small[0])

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
train size: 43410
validation size: 5426
test size: 5427
num_labels: 28
example labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment']
sample row: {'text': 'I would say they do because it was a horrible accident but they were also in the wrong. The military tried to get them to disperse they didnt.', 'labels': [9, 25], 'id': 'eehq2an'}


In [ ]:
def to_multihot(example):
    y = np.zeros(num_labels, dtype=np.float32)
    for lab in example["labels"]:
        y[lab] = 1.0
    example["labels"] = y.tolist()   # keeping as the float list as it stable in HF datasets
    return example

train_small = train_small.map(to_multihot)
val_small   = val_small.map(to_multihot)
test_small  = test_small.map(to_multihot)

print("labels length:", len(train_small[0]["labels"]))
print("nonzero labels idx:", [i for i, v in enumerate(train_small[0]["labels"]) if v == 1.0])
print("first 5 label values:", train_small[0]["labels"][:5])

labels length: 28
nonzero labels idx: [9, 25]
first 5 label values: [0, 0, 0, 0, 0]


## 3. Model & Training
- DistilBERT tokenizer + model
- Trainer API
- Evaluate with accuracy + macro F1

In [4]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_tok = train_small.map(tokenize_fn, batched=True)
val_tok   = val_small.map(tokenize_fn, batched=True)
test_tok  = test_small.map(tokenize_fn, batched=True)

cols = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch", columns=cols)
test_tok.set_format(type="torch", columns=cols)

print("tokenized keys:", train_tok[0].keys())
print("labels tensor dtype:", train_tok[0]["labels"].dtype, "shape:", train_tok[0]["labels"].shape)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

tokenized keys: dict_keys(['labels', 'input_ids', 'attention_mask'])
labels tensor dtype: torch.int64 shape: torch.Size([28])


In [ ]:
class MultiLabelDataCollator:
    def __init__(self, tokenizer):
        self.base = DataCollatorWithPadding(tokenizer=tokenizer)

    def __call__(self, features):
        batch = self.base(features)
        batch["labels"] = batch["labels"].float()  # Critical for BCEWithLogitsLoss
        return batch

data_collator = MultiLabelDataCollator(tokenizer)

tmp = data_collator([train_tok[i] for i in range(4)])
print("batch labels dtype:", tmp["labels"].dtype, "shape:", tmp["labels"].shape)

batch labels dtype: torch.float32 shape: torch.Size([4, 28])


In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.5).astype(np.float32)

    labels = labels.astype(np.float32)

    acc = (preds == labels).mean()
    f1  = f1_score(labels, preds, average="macro", zero_division=0)

    return {"accuracy": acc, "macro_f1": f1}

print("model num_labels:", model.config.num_labels)
print("problem_type:", model.config.problem_type)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model num_labels: 28
problem_type: multi_label_classification


In [7]:
training_args = TrainingArguments(
    output_dir=os.path.join(RESULTS_DIR, "goemotions_distilbert"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=32,
    num_train_epochs=5,              
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer ready")

Trainer ready


In [8]:
train_result = trainer.train()
print("train done")

val_metrics = trainer.evaluate()
print("val metrics:", val_metrics)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.091854,0.090700,0.968482,0.281723
2,0.090266,0.088251,0.969107,0.409921
3,0.064953,0.092470,0.969518,0.431077
4,0.043012,0.100609,0.968536,0.442928
5,0.043501,0.104305,0.967464,0.454927


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


train done


val metrics: {'eval_loss': 0.10430506616830826, 'eval_accuracy': 0.9674642857142857, 'eval_macro_f1': 0.45492713503647586, 'eval_runtime': 38.096, 'eval_samples_per_second': 52.499, 'eval_steps_per_second': 1.654, 'epoch': 5.0}


## 4. Save Outputs
- Save metrics to: E:\JYU\NLP\NLP\Project\results\
- Save label mapping for later reuse

In [9]:
# Save label mapping + metrics
label_map_path = os.path.join(RESULTS_DIR, "goemotions_label_names.json")
metrics_path   = os.path.join(RESULTS_DIR, "goemotions_val_metrics.json")

with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump(label_names, f, indent=2)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(val_metrics, f, indent=2)

print("saved:", label_map_path)
print("saved:", metrics_path)

saved: E:\JYU\NLP\NLP\Project\results\goemotions_label_names.json
saved: E:\JYU\NLP\NLP\Project\results\goemotions_val_metrics.json


In [10]:
test_metrics = trainer.evaluate(test_tok)
print("test metrics:", test_metrics)

test_metrics_path = os.path.join(RESULTS_DIR, "goemotions_test_metrics.json")
with open(test_metrics_path, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

print("saved:", test_metrics_path)

test metrics: {'eval_loss': 0.10408978164196014, 'eval_accuracy': 0.9671964285714286, 'eval_macro_f1': 0.4504067748172576, 'eval_runtime': 37.4626, 'eval_samples_per_second': 53.387, 'eval_steps_per_second': 1.682, 'epoch': 5.0}
saved: E:\JYU\NLP\NLP\Project\results\goemotions_test_metrics.json


In [ ]:
import os

EMOTION_MODEL_DIR = r"E:\JYU\NLP\NLP\Project\results\goemotions_distilbert"
os.makedirs(EMOTION_MODEL_DIR, exist_ok=True)

# Save model + tokenizer
trainer.save_model(EMOTION_MODEL_DIR)

tokenizer.save_pretrained(EMOTION_MODEL_DIR)

print("Saved model to:", EMOTION_MODEL_DIR)
print("Files now:", os.listdir(EMOTION_MODEL_DIR))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to: E:\JYU\NLP\NLP\Project\results\goemotions_distilbert
Files now: ['checkpoint-10000', 'checkpoint-1250', 'checkpoint-15000', 'checkpoint-20000', 'checkpoint-2500', 'checkpoint-25000', 'checkpoint-5000', 'config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


## Combined Analysis with LIAR

In [12]:
import torch
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
import json
import os

EMOTION_MODEL_DIR = r"E:\JYU\NLP\NLP\Project\results\goemotions_distilbert"
LABEL_NAMES_PATH = r"E:\JYU\NLP\NLP\Project\results\goemotions_label_names.json"

# Load label names
with open(LABEL_NAMES_PATH, "r", encoding="utf-8") as f:
    emotion_labels = json.load(f)

num_emotions = len(emotion_labels)

tokenizer_emotion = DistilBertTokenizerFast.from_pretrained(EMOTION_MODEL_DIR)

model_emotion = DistilBertForSequenceClassification.from_pretrained(
    EMOTION_MODEL_DIR,
    num_labels=num_emotions
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_emotion.to(device)
model_emotion.eval()

print("Loaded emotion model.")
print("Number of emotion labels:", num_emotions)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loaded emotion model.
Number of emotion labels: 28


In [13]:
from datasets import load_dataset

data_files = {
    "train": r"E:\JYU\NLP\NLP\Project\data\liar\train.tsv",
    "validation": r"E:\JYU\NLP\NLP\Project\data\liar\valid.tsv",
    "test": r"E:\JYU\NLP\NLP\Project\data\liar\test.tsv",
}

column_names = [
    "id", "label", "statement", "subject", "speaker", "speaker_job",
    "state", "party", "barely_true_counts", "false_counts",
    "half_true_counts", "mostly_true_counts", "pants_on_fire_counts",
    "context"
]

ds_liar = load_dataset(
    "csv",
    data_files=data_files,
    delimiter="\t",
    column_names=column_names,
)

TRUE_LABELS = {"half-true", "mostly-true", "true"}
FALSE_LABELS = {"pants-fire", "false", "barely-true"}

def to_binary(example):
    lab = example["label"]
    if lab in TRUE_LABELS:
        example["binary_label"] = 1
    else:
        example["binary_label"] = 0
    return example

ds_liar = ds_liar.map(to_binary)

print(ds_liar)
print(ds_liar["train"][0]["statement"], "=>", ds_liar["train"][0]["binary_label"])

DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'speaker_job', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'binary_label'],
        num_rows: 10240
    })
    validation: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'speaker_job', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'binary_label'],
        num_rows: 1284
    })
    test: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'speaker_job', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'binary_label'],
        num_rows: 1267
    })
})
Says the Annies List political group supports third-trimester abortions on demand. => 0


In [14]:
import os, json
import numpy as np
import torch

OUT_DIR = r"E:\JYU\NLP\NLP\Project\results"
os.makedirs(OUT_DIR, exist_ok=True)

texts = ds_liar["train"]["statement"]
bin_labels = np.array(ds_liar["train"]["binary_label"])

# clean
clean_texts, clean_bin = [], []
for t, y in zip(texts, bin_labels):
    if isinstance(t, str) and t.strip():
        clean_texts.append(t)
        clean_bin.append(int(y))

clean_bin = np.array(clean_bin)

batch_size = 64
all_probs = []

model_emotion.eval()
for i in range(0, len(clean_texts), batch_size):
    batch = clean_texts[i:i+batch_size]
    enc = tokenizer_emotion(
        batch, truncation=True, padding=True, max_length=128, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = model_emotion(**enc).logits
        probs = torch.sigmoid(logits).cpu().numpy()
    all_probs.append(probs)

probs = np.vstack(all_probs)

top1 = probs.argmax(axis=1)
top1_names = [emotion_labels[i] for i in top1]

from collections import Counter
fake_top1 = [t for t, y in zip(top1_names, clean_bin) if y == 0]
true_top1 = [t for t, y in zip(top1_names, clean_bin) if y == 1]

top_fake_top15 = Counter(fake_top1).most_common(15)
top_trueish_top15 = Counter(true_top1).most_common(15)

fake_mean = probs[clean_bin == 0].mean(axis=0)
true_mean = probs[clean_bin == 1].mean(axis=0)
diff = true_mean - fake_mean

top_more_true = np.argsort(diff)[-10:][::-1].tolist()
top_more_fake = np.argsort(diff)[:10].tolist()

summary = {
    "n_total": int(len(texts)),
    "n_clean": int(len(clean_texts)),
    "fake_count": int((clean_bin==0).sum()),
    "trueish_count": int((clean_bin==1).sum()),
    "top1_fake_top15": top_fake_top15,
    "top1_trueish_top15": top_trueish_top15,
    "top10_more_trueish": [
        {"emotion": emotion_labels[i], "trueish_mean": float(true_mean[i]), "fake_mean": float(fake_mean[i]), "diff": float(diff[i])}
        for i in top_more_true
    ],
    "top10_more_fake": [
        {"emotion": emotion_labels[i], "trueish_mean": float(true_mean[i]), "fake_mean": float(fake_mean[i]), "diff": float(diff[i])}
        for i in top_more_fake
    ],
}

out_path = os.path.join(OUT_DIR, "liar_emotion_vs_truthfulness_train_summary.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:", out_path)
print("Counts:", summary["fake_count"], summary["trueish_count"])
print("Top3 more true-ish:", [x["emotion"] for x in summary["top10_more_trueish"][:3]])
print("Top3 more fake:", [x["emotion"] for x in summary["top10_more_fake"][:3]])

Saved: E:\JYU\NLP\NLP\Project\results\liar_emotion_vs_truthfulness_train_summary.json
Counts: 4488 5752
Top3 more true-ish: ['approval', 'admiration', 'realization']
Top3 more fake: ['neutral', 'annoyance', 'disapproval']
